# Experiment Tracking

In [1]:
!pip install torchinfo
!pip install torchmetrics
!pip install tensorboard

In [2]:
import torch
import torch.nn as nn
import requests
import zipfile
from pathlib import Path
import os
import torchvision
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import torchvision.models as Models
from torchinfo import summary
from tqdm.auto import tqdm
import torchmetrics
from torchmetrics.classification import MulticlassAccuracy
from torch.utils.tensorboard import SummaryWriter

## Get data

In [3]:
def get_data(
        raw_link: str,
        save_folder: Path,
        zip_file: Path
):
    
    save_folder.mkdir(parents=True, exist_ok=True)

    if not zip_file.is_file():
        request = requests.get(raw_link)
        with open(zip_file, 'wb') as f:
            f.write(request.content)
        
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall(save_folder)
        
    print(f'Confirming downloaded data for {str(save_folder)}: ')
    print(os.listdir(save_folder))
    print(os.listdir(save_folder / 'train'))
    print(os.listdir(save_folder / 'train' / 'sushi'))
    print('\n')


data_folder = Path('data')
data_folder.mkdir(parents=True, exist_ok=True)
save_folder_10pc = data_folder / '10pc'
save_folder_20pc = data_folder / '20pc'

# Get 10 % data at ./data/10pc/train...
get_data(
    raw_link = 'https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip',
    save_folder = save_folder_10pc,
    zip_file = data_folder / 'pizza_sushi_steak_10pc.zip'
)

get_data(
    raw_link = 'https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi_20_percent.zip',
    save_folder = save_folder_20pc,
    zip_file = data_folder / 'pizza_sushi_steak_20pc.zip'
)

    


Confirming downloaded data for data/10pc: 
['train', 'test']
['sushi', 'pizza', 'steak']
['390178.jpg', '2873571.jpg', '3360207.jpg', '542188.jpg', '2017378.jpg', '1214108.jpg', '3004029.jpg', '840444.jpg', '385154.jpg', '2797464.jpg', '710379.jpg', '1615453.jpg', '1221830.jpg', '2674024.jpg', '929471.jpg', '1138695.jpg', '1571146.jpg', '2980779.jpg', '93139.jpg', '2021381.jpg', '686426.jpg', '497686.jpg', '424994.jpg', '2175561.jpg', '2004525.jpg', '2720223.jpg', '794647.jpg', '1575445.jpg', '268990.jpg', '1129338.jpg', '2323548.jpg', '307738.jpg', '3107839.jpg', '821108.jpg', '773725.jpg', '3426958.jpg', '765684.jpg', '121940.jpg', '2574453.jpg', '14046.jpg', '148799.jpg', '377047.jpg', '700405.jpg', '1070104.jpg', '3353428.jpg', '2021685.jpg', '1551817.jpg', '843815.jpg', '2641778.jpg', '1232045.jpg', '2120573.jpg', '855721.jpg', '17704.jpg', '3737197.jpg', '169392.jpg', '2590819.jpg', '1552504.jpg', '3360232.jpg', '3579071.jpg', '200025.jpg', '2019344.jpg', '1209865.jpg', '1957449.

## Create dataset

In [4]:
def create_dataset(
        train_dir: Path,
        test_dir: Path,
        transforms: torchvision.transforms,
        verbose: bool = False
) -> tuple[torchvision.datasets, torchvision.datasets]:
    
    train_data = ImageFolder(
        root = train_dir,
        transform = transforms
    )

    test_data = ImageFolder(
        root = test_dir,
        transform = transforms
    )

    if verbose:
        img, label = train_data[0]
        print('\nConfirming created dataset: ')
        print('Total number of data in training dataset: ', len(train_data))
        print('Shape of an image: ', img.shape)
        print(f'Range of value in an image: {torch.min(img)} to {torch.max(img)}')
        print('Available classes: ', train_data.classes)
        print(f'Example of an image label: {label}: {train_data.classes[label]}')

    return train_data, test_data

## Create Dataloader

In [5]:
def create_dataloader(
        train_data: torchvision.datasets,
        test_data: torchvision.datasets,
        batch_size: int,
        verbose: bool = False
) -> tuple[DataLoader, DataLoader]:
    
    train_dataloader = DataLoader(
        dataset = train_data,
        batch_size = batch_size,
        shuffle = True
    )

    test_dataloader = DataLoader(
        dataset = test_data,
        batch_size = batch_size,
        shuffle = False
    )

    if verbose:
        img_batch, label_batch = next(iter(train_dataloader))
        print('\nConfirming created dataloaders: ')
        print('Total number of batches: ', len(train_dataloader))
        print('Size of a batch of images: ', img_batch.shape)
        print('Size of a batch of labels: ', label_batch.shape)

    return train_dataloader, test_dataloader

## Create Model

In [6]:
def create_model(
        model_idx: int,
        is_pretrained: bool,
        is_frozen: bool,
        num_classes: int,
        verbose: bool = False
)-> tuple[nn.Module, torchvision.transforms, str]:
    
    model_count = 2
    model_types = ['EffB0', 'EffB2']
    in_features = [1280 ,1408]
    test_img_sizes = [224, 288]
    if model_idx >= model_count:
        raise ValueError(f'Only accept model_idx with value of less than {model_count}.')

    weights = 'DEFAULT' if is_pretrained else None

    model_type = model_types[model_idx]
    model, transforms = None, None
    if model_idx == 0:
        model = Models.efficientnet_b0(weights=weights)
        transforms = Models.EfficientNet_B0_Weights.IMAGENET1K_V1.transforms()
        
    elif model_idx == 1:
        model = Models.efficientnet_b2(weights=weights)
        transforms = Models.EfficientNet_B2_Weights.IMAGENET1K_V1.transforms()
    
    if is_frozen:
            for params in model.parameters():
                params.requires_grad = False
            model.classifier = nn.Sequential(
                nn.Dropout(p=0.3, inplace=True),
                nn.Linear(in_features=in_features[model_idx], out_features=num_classes, bias=True)
            )
        
    
    
    

    test_img_size = test_img_sizes[model_idx]
    if verbose:
        print(f'\nConfirming the created model with an input of (1, 3, {test_img_size}, {test_img_size}): ')
        print(summary(
            model, 
            input_size=(1, 3, test_img_size, test_img_size),
            col_names=['input_size', 'output_size', 'num_params', 'trainable'],
            row_settings=['var_names']
            ))
    
    return model, transforms, model_type
    

## Train

In [7]:
def train(
        model: nn.Module,
        model_type: str,
        data_type: str,
        optimizer: torch.optim,
        loss_fn: nn.Module,
        acc_fn: torchmetrics.classification,
        epochs: int,
        device: torch.device,
        train_dataloader: DataLoader,
        test_dataloader: DataLoader,
        num_classes: int
) -> str:
    model_name = f'{model_type}_{data_type}_{epochs}'
    run_name = f'runs/{model_name}'
    writer = SummaryWriter(run_name)

    print('\n---------------------------------------------')
    print('Begining the training with these variations: ')
    print('Model Type: ', model_type)
    print('Data Type: ', data_type)
    print('Epochs: ', epochs)


    model = model.to(device)

    for epoch in tqdm(range(epochs)):
        ep_train_loss = 0
        ep_train_acc = 0
        ep_test_loss = 0
        ep_test_acc = 0

        model.train()
        for X, y in train_dataloader:
            X, y = X.to(device), y.to(device)
            y_preds = model(X)
            train_loss = loss_fn(y_preds, torch.nn.functional.one_hot(y, num_classes=num_classes).to(torch.float32))
            train_acc = acc_fn(torch.argmax(y_preds, dim=1), y)

            optimizer.zero_grad()
            train_loss.backward()
            optimizer.step()

            ep_train_loss += train_loss.item()
            ep_train_acc += train_acc.item()

        model.eval()
        with torch.inference_mode():
            for X, y in test_dataloader:
                X, y = X.to(device), y.to(device)
                y_preds = model(X)
                test_loss = loss_fn(y_preds, torch.nn.functional.one_hot(y, num_classes=num_classes).to(torch.float32))
                test_acc = acc_fn(torch.argmax(y_preds, dim=1), y)

                ep_test_loss += test_loss.item()
                ep_test_acc += test_acc.item()
        
        num_train_batches = len(train_dataloader)
        num_test_batches = len(test_dataloader)
        ep_train_loss /= num_train_batches
        ep_train_acc /= num_train_batches
        ep_test_loss /= num_test_batches
        ep_test_acc /= num_test_batches

        writer.add_scalars(
            'Loss',
            {
                'train_loss': ep_train_loss,
                'test_loss': ep_test_loss
            },
            epoch
        )

        writer.add_scalars(
            'Accuracy',
            {
                'train_acc': ep_train_acc,
                'test_acc': ep_test_acc
            },
            epoch
        )

        print('Epoch: ', epoch)
        print(f'Train Loss: {ep_train_loss:.4f} | Train Acc: {ep_train_acc:.2f}')
        print(f'Test Loss: {ep_test_loss:.4f} | Test Acc: {ep_test_acc:.2f}')
    
    writer.flush()
    writer.close()

    save_folder = Path('models')
    save_folder.mkdir(parents=True, exist_ok=True)
    save_path = save_folder / f'{model_name}.pt'
    torch.save(model.state_dict(), save_path)

    print('\n---End of experiment---')
    print(f'The model is saved at {save_path}')
    print(f'The run data is saved at {run_name}')
    print('\n---------------------------------------------')
    return model_name
        

                
        
         

## Experiments

In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
NUM_CLASSES = len(os.listdir(save_folder_10pc / 'train'))
BATCH_SIZE = 32
LEARNING_RATE = 1E-3
EPOCHS = 5

model, transform, model_type =  create_model(
    model_idx = 0, # Need to Change
    is_pretrained= True,
    is_frozen = True,
    num_classes = NUM_CLASSES,
    verbose = False
)

train_data, test_data = create_dataset(
    train_dir = save_folder_10pc / 'train', # Need to change
    test_dir = save_folder_10pc / 'test', # Need to change
    transforms = transform,
    verbose = False
)

train_dataloader, test_dataloader = create_dataloader(
    train_data = train_data,
    test_data = test_data,
    batch_size = BATCH_SIZE,
    verbose = False
)

optimizer = torch.optim.Adam(
    model.parameters(), LEARNING_RATE
)

loss_fn = nn.CrossEntropyLoss()

acc_fn = MulticlassAccuracy(
    num_classes = NUM_CLASSES,
    average = 'micro'
).to(device)

model_name = train(
    model = model,
    model_type = model_type,
    data_type = 'data10pc',# Need to change
    optimizer = optimizer,
    loss_fn = loss_fn,
    acc_fn = acc_fn,
    epochs = EPOCHS,
    device = device,
    train_dataloader= train_dataloader,
    test_dataloader=test_dataloader,
    num_classes = NUM_CLASSES
)




---------------------------------------------
Begining the training with these variations: 
Model Type:  EffB0
Data Type:  data10pc
Epochs:  5


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch:  0
Train Loss: 1.0528 | Train Acc: 0.46
Test Loss: 0.8622 | Test Acc: 0.73
Epoch:  1
Train Loss: 0.8689 | Train Acc: 0.71
Test Loss: 0.7586 | Test Acc: 0.85
Epoch:  2
Train Loss: 0.7088 | Train Acc: 0.85
Test Loss: 0.7108 | Test Acc: 0.81
Epoch:  3
Train Loss: 0.7140 | Train Acc: 0.72
Test Loss: 0.6553 | Test Acc: 0.87
Epoch:  4
Train Loss: 0.6058 | Train Acc: 0.88
Test Loss: 0.5661 | Test Acc: 0.94

---End of experiment---
The model is saved at models/EffB0_data10pc_5.pt
The run data is saved at runs/EffB0_data10pc_5

---------------------------------------------


## Tensorboar
* Only works on colab

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

Reusing TensorBoard on port 6007 (pid 44304), started 0:05:20 ago. (Use '!kill 44304' to kill it.)

<IPython.core.display.Javascript object>